In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
CANONICAL_DIR = PROJECT_ROOT / "data/canonical"

# Inventory every canonical CSV without modifying any files.
inventory = []

for path in sorted(CANONICAL_DIR.glob("*.csv")):
    df = pd.read_csv(path)

    inventory.append({
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "seasons": df["season"].nunique() if "season" in df else None,
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

print("TOTAL CANONICAL CSV FILES:", len(inventory_df))
print("TOTAL CANONICAL RECORDS:", inventory_df["rows"].sum())

,file,rows,columns,seasons,missing_cells,duplicate_rows
0,franchise_season_aliases.csv,776,5,26.0,0,0
1,franchises.csv,30,3,NaN,0,0
2,leadership_tenures.csv,251,8,NaN,532,0
3,minimum_salary_rules.csv,286,5,26.0,0,0
4,player_awards.csv,2570,24,26.0,30458,0
5,player_salaries.csv,11826,6,26.0,0,0
6,player_salaries_unresolved.csv,380,6,26.0,436,0
7,player_team_seasons.csv,12810,49,26.0,82,0
8,players.csv,2584,7,NaN,463,0
9,rookie_salary_scale.csv,775,8,26.0,0,0


TOTAL CANONICAL CSV FILES: 12
TOTAL CANONICAL RECORDS: 33090


In [2]:
# --------------------------------------------------
# CANONICAL INTEGRITY AUDIT — KEYS & REFERENCES
# --------------------------------------------------

import pandas as pd

# Load canonical tables.
tables = {
    path.stem: pd.read_csv(path)
    for path in CANONICAL_DIR.glob("*.csv")
}

# Inspect identifiers before defining the checks.
for name in [
    "franchises",
    "players",
    "seasons",
    "team_seasons",
    "player_team_seasons",
    "player_salaries",
    "player_awards",
    "leadership_tenures"
]:
    print(f"\n{name}")
    print(tables[name].columns.tolist())


franchises
['franchise_id', 'current_team_name', 'current_abbreviation']

players
['player_id', 'player_name', 'country', 'college', 'draft_year', 'draft_round', 'draft_number']

seasons
['season', 'Year', 'Salary Cap', 'Salary Cap Floor Pct', 'Salary Cap Floor', 'Luxury Tax', '1st Apron', '2nd Apron', 'BAE', 'Non-Taxpayer MLE', 'Taxpayer MLE', 'Team Room MLE', 'cba_id']

team_seasons
['team_name', 'wins', 'losses', 'win_pct', 'games_behind', 'points_per_game', 'opponent_points_per_game', 'simple_rating_system', 'conference', 'season', 'franchise_id', 'league_rank', 'average_age', 'pythagorean_wins', 'pythagorean_losses', 'margin_of_victory', 'strength_of_schedule', 'offensive_rating', 'defensive_rating', 'net_rating', 'pace', 'free_throw_rate', 'three_point_attempt_rate', 'true_shooting_pct', 'off_eFG%', 'off_TOV%', 'off_ORB%', 'off_FT/FGA', 'def_eFG%', 'def_TOV%', 'def_DRB%', 'def_FT/FGA', 'arena', 'attendance', 'attendance_per_game', 'seed', 'playoff_series_won', 'round_score', 'fu

In [3]:
# --------------------------------------------------
# FINAL AUDIT — PRIMARY KEYS AND FOREIGN KEYS
# --------------------------------------------------

import pandas as pd

# 1. Define expected unique keys.
primary_keys = {
    "franchises": ["franchise_id"],
    "players": ["player_id"],
    "seasons": ["season"],
    "franchise_season_aliases": ["franchise_id", "season"],
    "team_seasons": ["franchise_id", "season"],
    "player_team_seasons": ["player_id", "franchise_id", "season"],
    "player_salaries": ["player_id", "franchise_id", "season"],
    "rookie_salary_scale": ["season", "draft_pick"],
    "minimum_salary_rules": ["season", "years_of_service"]
}

print("PRIMARY KEY VALIDATION")
print("-" * 55)

for name, keys in primary_keys.items():
    df = tables[name]

    duplicate_count = df.duplicated(keys).sum()
    null_key_count = df[keys].isna().any(axis=1).sum()

    status = (
        "PASS"
        if duplicate_count == 0 and null_key_count == 0
        else "FAIL"
    )

    print(
        f"{name:30} {status:5} "
        f"duplicates={duplicate_count} "
        f"null_keys={null_key_count}"
    )


# 2. Foreign-key validation.
# Compare actual identifiers against their parent tables.

def check_fk(child_name, child_col, parent_name, parent_col):
    child = tables[child_name]
    parent = tables[parent_name]

    missing = (
        set(child[child_col].dropna())
        - set(parent[parent_col].dropna())
    )

    print(
        f"{child_name}.{child_col} → "
        f"{parent_name}.{parent_col}: "
        f"{'PASS' if not missing else 'FAIL'} "
        f"({len(missing)} unmatched IDs)"
    )

    if missing:
        print("  Examples:", list(missing)[:10])


print("\nFOREIGN KEY VALIDATION")
print("-" * 55)

checks = [
    ("franchise_season_aliases", "franchise_id", "franchises", "franchise_id"),
    ("team_seasons", "franchise_id", "franchises", "franchise_id"),
    ("player_team_seasons", "franchise_id", "franchises", "franchise_id"),
    ("player_salaries", "franchise_id", "franchises", "franchise_id"),
    ("leadership_tenures", "franchise_id", "franchises", "franchise_id"),

    ("player_team_seasons", "player_id", "players", "player_id"),
    ("player_salaries", "player_id", "players", "player_id"),
    ("player_awards", "player_id", "players", "player_id"),

    ("franchise_season_aliases", "season", "seasons", "season"),
    ("team_seasons", "season", "seasons", "season"),
    ("player_team_seasons", "season", "seasons", "season"),
    ("player_salaries", "season", "seasons", "season"),
    ("player_awards", "season", "seasons", "season"),
    ("rookie_salary_scale", "season", "seasons", "season"),
    ("minimum_salary_rules", "season", "seasons", "season")
]

for check in checks:
    check_fk(*check)


# 3. Composite franchise-season relationships.
# A valid franchise ID and valid season independently
# do not guarantee the franchise existed in that season.

print("\nFRANCHISE-SEASON RELATIONSHIPS")
print("-" * 55)

valid_team_keys = pd.MultiIndex.from_frame(
    tables["team_seasons"][["franchise_id", "season"]]
)

for name in [
    "franchise_season_aliases",
    "player_team_seasons",
    "player_salaries"
]:
    df = tables[name]

    child_keys = pd.MultiIndex.from_frame(
        df[["franchise_id", "season"]]
    )

    unmatched = ~child_keys.isin(valid_team_keys)

    print(
        f"{name:30} "
        f"{'PASS' if unmatched.sum() == 0 else 'REVIEW'} "
        f"unmatched_rows={unmatched.sum()}"
    )

    if unmatched.any():
        display(
            df.loc[
                unmatched,
                ["franchise_id", "season"]
            ].drop_duplicates().head(10)
        )

PRIMARY KEY VALIDATION
-------------------------------------------------------
franchises                     PASS  duplicates=0 null_keys=0
players                        PASS  duplicates=0 null_keys=0
seasons                        PASS  duplicates=0 null_keys=0
franchise_season_aliases       PASS  duplicates=0 null_keys=0
team_seasons                   PASS  duplicates=0 null_keys=0
player_team_seasons            PASS  duplicates=0 null_keys=0
player_salaries                PASS  duplicates=0 null_keys=0
rookie_salary_scale            PASS  duplicates=0 null_keys=0
minimum_salary_rules           PASS  duplicates=0 null_keys=0

FOREIGN KEY VALIDATION
-------------------------------------------------------
franchise_season_aliases.franchise_id → franchises.franchise_id: PASS (0 unmatched IDs)
team_seasons.franchise_id → franchises.franchise_id: PASS (0 unmatched IDs)
player_team_seasons.franchise_id → franchises.franchise_id: PASS (0 unmatched IDs)
player_salaries.franchise_id → franc

In [4]:
# --------------------------------------------------
# FINAL AUDIT — ANALYTICAL READINESS
# --------------------------------------------------

import pandas as pd

awards = tables["player_awards"]
leaders = tables["leadership_tenures"]
salaries = tables["player_salaries"]
player_seasons = tables["player_team_seasons"]
teams = tables["team_seasons"]

# 1. Award observations and leadership tenures
print("AWARD COVERAGE")
display(
    awards.groupby(["award_type", "source_table"], dropna=False)
    .agg(
        records=("player_id", "size"),
        players=("player_id", "nunique"),
        seasons=("season", "nunique")
    )
    .reset_index()
)

print("\nPOTENTIAL DUPLICATE AWARD OBSERVATIONS")
award_duplicates = awards[
    awards.duplicated(
        ["player_id", "season", "award_type", "source_table"],
        keep=False
    )
]
print("Rows requiring review:", len(award_duplicates))
display(award_duplicates.head(10))

print("\nLEADERSHIP COVERAGE")
display(
    leaders.groupby(["role", "source_table"], dropna=False)
    .agg(
        records=("franchise_id", "size"),
        franchises=("franchise_id", "nunique")
    )
    .reset_index()
)

print("\nPOTENTIAL DUPLICATE LEADERSHIP TENURES")
leader_duplicates = leaders[
    leaders.duplicated(
        ["franchise_id", "leader_name", "role", "start_date"],
        keep=False
    )
]
print("Rows requiring review:", len(leader_duplicates))
display(leader_duplicates.head(10))


# 2. Player-salary coverage
print("\nPLAYER-SALARY COVERAGE")

salary_keys = salaries[
    ["player_id", "franchise_id", "season"]
].drop_duplicates()

roster_coverage = player_seasons[
    ["player_id", "franchise_id", "season"]
].merge(
    salary_keys.assign(has_salary=True),
    on=["player_id", "franchise_id", "season"],
    how="left",
    validate="one_to_one"
)

print("Player-team-season records:", len(roster_coverage))
print("Records with matching salary:",
      roster_coverage["has_salary"].eq(True).sum())
print("Records without matching salary:",
      roster_coverage["has_salary"].isna().sum())

print("\nSALARY COVERAGE BY SEASON")
display(
    roster_coverage.assign(
        has_salary=roster_coverage["has_salary"].fillna(False)
    )
    .groupby("season")["has_salary"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "roster_records",
        "sum": "matched_salaries",
        "mean": "salary_match_rate"
    })
    .sort_index()
)


# 3. S18 coverage and consistency
print("\nLUXURY-TAX COVERAGE")

display(
    teams.groupby("season")
    .agg(
        teams=("franchise_id", "size"),
        s18_records=("has_luxury_tax_source", "sum"),
        actual_tax_records=("tax_paid", "count"),
        taxable_payroll_records=("taxable_payroll", "count"),
        estimated_tax_records=("estimated_tax_bill", "count")
    )
    .sort_index()
)

print("\nTAX SPACE CHECK")

tax_rows = teams.dropna(
    subset=["s18_tax_threshold", "taxable_payroll", "tax_space"]
)

tax_space_errors = (
    tax_rows["tax_space"] -
    (tax_rows["s18_tax_threshold"] - tax_rows["taxable_payroll"])
).abs().gt(1).sum()

print("Comparable tax-space records:", len(tax_rows))
print("Tax-space inconsistencies:", tax_space_errors)

AWARD COVERAGE


,award_type,source_table,records,players,seasons
0,all_defensive,defense_history.csv,263,92,26
1,all_nba,S14A_all_nba.csv,390,112,26
2,all_rookie,rookie_history.csv,260,260,26
3,all_star,s14c_all_star_selections_2001_2026.csv,681,178,26
4,dpoy,dpoy_history.csv,426,153,26
5,mvp,mvp_history.csv,352,102,26
6,roy,roy_history.csv,198,198,26



POTENTIAL DUPLICATE AWARD OBSERVATIONS
Rows requiring review: 0


,season,source_player_name,player_position,player_id,award_type,source_table,team_selection,age,team,points_won,...,voting_rank,first_place_votes,award_winner,first_team_votes,second_team_votes,all_star_appearances,votes,selection_type,player_status,conference



LEADERSHIP COVERAGE


...


PLAYER-SALARY COVERAGE
Player-team-season records: 12810
Records with matching salary: 10601
Records without matching salary: 2209

SALARY COVERAGE BY SEASON


,roster_records,matched_salaries,salary_match_rate
season,,,
2000-2001,441,369,0.836735
2001-2002,440,411,0.934091
2002-2003,428,278,0.649533
2003-2004,442,177,0.400452
2004-2005,464,118,0.25431
2005-2006,458,420,0.917031
2006-2007,458,432,0.943231
2007-2008,451,429,0.95122
2008-2009,445,427,0.959551



LUXURY-TAX COVERAGE


,teams,s18_records,actual_tax_records,taxable_payroll_records,estimated_tax_records
season,,,,,
2000-2001,29,0,0,0,0
2001-2002,29,0,0,0,0
2002-2003,29,29,29,0,0
2003-2004,29,29,29,0,0
2004-2005,30,30,30,0,0
2005-2006,30,30,30,0,0
2006-2007,30,30,30,0,0
2007-2008,30,30,30,0,0
2008-2009,30,30,30,0,0



TAX SPACE CHECK
Comparable tax-space records: 450
Tax-space inconsistencies: 0


In [5]:
# Summarize salary coverage by season
salary_summary = (
    roster_coverage
    .assign(has_salary=roster_coverage["has_salary"].eq(True))
    .groupby("season")
    .agg(
        roster_records=("has_salary", "size"),
        matched=("has_salary", "sum")
    )
)

salary_summary["missing"] = (
    salary_summary["roster_records"] - salary_summary["matched"]
)

salary_summary["match_pct"] = (
    100 * salary_summary["matched"] / salary_summary["roster_records"]
).round(1)

# Show only the five weakest seasons
display(salary_summary.nsmallest(5, "match_pct"))

print(
    "\nOverall salary match rate:",
    round(100 * 10601 / 12810, 1),
    "%"
)

,roster_records,matched,missing,match_pct
season,,,,
2004-2005,464,118,346,25.4
2003-2004,442,177,265,40.0
2002-2003,428,278,150,65.0
2017-2018,540,402,138,74.4
2021-2022,605,461,144,76.2



Overall salary match rate: 82.8 %


In [6]:
# --------------------------------------------------
# SALARY MATCH DIAGNOSTIC
# --------------------------------------------------

keys = ["player_id", "franchise_id", "season"]

roster = tables["player_team_seasons"][keys].copy()
salary = tables["player_salaries"][keys].copy()

# Find roster observations without an exact salary match.
missing = roster.merge(
    salary,
    on=keys,
    how="left",
    indicator=True,
    validate="one_to_one"
)

missing = missing[
    missing["_merge"] == "left_only"
][keys].copy()

# Check whether those players have a salary recorded
# somewhere else in the SAME season.
salary_player_season = salary[
    ["player_id", "season"]
].drop_duplicates()

diagnostic = missing.merge(
    salary_player_season.assign(
        salary_exists_other_team=True
    ),
    on=["player_id", "season"],
    how="left",
    validate="many_to_one"
)

diagnostic["salary_exists_other_team"] = (
    diagnostic["salary_exists_other_team"]
    .fillna(False)
    .astype(bool)
)

other_team = diagnostic["salary_exists_other_team"].sum()
no_salary = (~diagnostic["salary_exists_other_team"]).sum()

print("TOTAL UNMATCHED ROSTER RECORDS:", len(diagnostic))
print("Salary exists for player-season, but franchise differs:", other_team)
print("No salary found for player-season:", no_salary)

print("\nBREAKDOWN BY SEASON")

summary = (
    diagnostic
    .groupby("season")
    .agg(
        unmatched_roster_records=("player_id", "size"),
        different_franchise=("salary_exists_other_team", "sum")
    )
)

summary["no_player_season_salary"] = (
    summary["unmatched_roster_records"]
    - summary["different_franchise"]
)

display(
    summary.sort_values(
        "unmatched_roster_records",
        ascending=False
    ).head(10)
)

TOTAL UNMATCHED ROSTER RECORDS: 2209
Salary exists for player-season, but franchise differs: 96
No salary found for player-season: 2113

BREAKDOWN BY SEASON


,unmatched_roster_records,different_franchise,no_player_season_salary
season,,,
2004-2005,346,0,346
2003-2004,265,0,265
2002-2003,150,0,150
2021-2022,144,6,138
2017-2018,138,1,137
2025-2026,129,2,127
2023-2024,123,3,120
2024-2025,110,8,102
2019-2020,91,9,82


In [7]:
# Compare unresolved salary records with missing roster salaries.

unresolved = tables["player_salaries_unresolved"]

print("UNRESOLVED SALARY RECORDS BY SEASON")

display(
    unresolved.groupby("season")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nUNRESOLVED SALARY COLUMNS")
print(unresolved.columns.tolist())

print("\nSAMPLE UNRESOLVED RECORDS")
display(unresolved.head(5))

UNRESOLVED SALARY RECORDS BY SEASON


season
2017-2018    62
2012-2013    53
2016-2017    41
2011-2012    33
2010-2011    29
2015-2016    22
2008-2009    20
2007-2008    18
2014-2015    17
2009-2010    15
dtype: int64


UNRESOLVED SALARY COLUMNS
['player', 'team', 'season', 'salary', 'player_id', 'franchise_id']

SAMPLE UNRESOLVED RECORDS


,player,team,season,salary,player_id,franchise_id
0,Steve Smith,Portland Trail Blazers,2000-2001,8100000,NaN,1.610613e+09
1,Isaac Austin,Vancouver Grizzlies,2000-2001,5500000,NaN,1.610613e+09
2,Clarence Weatherspoon,Cleveland Cavaliers,2000-2001,2100000,NaN,1.610613e+09
3,Tyrone Bogues,New York Knicks,2000-2001,1740000,NaN,1.610613e+09
4,Steve Smith,San Antonio Spurs,2001-2002,9000000,NaN,1.610613e+09


In [8]:
# --------------------------------------------------
# CHECK RECOVERABLE UNRESOLVED SALARY RECORDS
# --------------------------------------------------

unresolved = tables["player_salaries_unresolved"].copy()
roster = tables["player_team_seasons"].copy()
players = tables["players"][["player_id", "player_name"]].copy()

# Add canonical player names to the roster.
roster_named = roster[
    ["player_id", "franchise_id", "season"]
].merge(
    players,
    on="player_id",
    how="left",
    validate="many_to_one"
)

# Normalize names to handle case and extra whitespace.
unresolved["name_key"] = (
    unresolved["player"].str.strip().str.casefold()
)

roster_named["name_key"] = (
    roster_named["player_name"].str.strip().str.casefold()
)

# Match on player name, season, and franchise.
candidates = unresolved.reset_index(names="unresolved_row").merge(
    roster_named[
        ["player_id", "franchise_id", "season", "name_key"]
    ],
    on=["franchise_id", "season", "name_key"],
    how="left",
    suffixes=("_source", "_candidate")
)

# Count distinct candidate player IDs per unresolved record.
candidate_counts = (
    candidates.groupby("unresolved_row")["player_id_candidate"]
    .nunique()
)

print("Total unresolved salary records:", len(unresolved))
print("Exactly one matching canonical player:",
      (candidate_counts == 1).sum())
print("Multiple matching players:",
      (candidate_counts > 1).sum())
print("No exact name/team/season match:",
      (candidate_counts == 0).sum())

print("\nRECOVERABLE MATCHES BY SEASON")

recoverable = unresolved.loc[
    candidate_counts[candidate_counts == 1].index
]

display(
    recoverable.groupby("season")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Total unresolved salary records: 380
Exactly one matching canonical player: 0
Multiple matching players: 0
No exact name/team/season match: 380

RECOVERABLE MATCHES BY SEASON


Series([], dtype: int64)

In [9]:
teams = tables["team_seasons"]

tax_rows = teams.dropna(
    subset=["s18_tax_threshold", "taxable_payroll", "tax_space"]
)

errors = (
    tax_rows["tax_space"]
    - (
        tax_rows["s18_tax_threshold"]
        - tax_rows["taxable_payroll"]
    )
).abs().gt(1).sum()

print("Comparable tax records:", len(tax_rows))
print("Tax-space inconsistencies:", errors)
print("S18 source records:", teams["has_luxury_tax_source"].sum())

Comparable tax records: 450
Tax-space inconsistencies: 0
S18 source records: 718


In [12]:
import pandas as pd

# Load the final canonical team-season dataset.
c04 = pd.read_csv("../data/canonical/team_seasons.csv")

# Identify columns associated with tax thresholds and tax payments.
tax_cols = [
    col for col in c04.columns
    if "tax" in col.lower() or "threshold" in col.lower()
]

# Display their names and data completeness.
print("C04 rows:", len(c04))
print("\nTax-related columns:")
print(tax_cols)

print("\nNon-null observations:")
print(c04[tax_cols].notna().sum())

C04 rows: 776

Tax-related columns:
['s18_tax_threshold', 'taxable_payroll', 'tax_space', 'tax_paid', 'estimated_tax_bill', 'has_luxury_tax_source']

Non-null observations:
s18_tax_threshold        688
taxable_payroll          450
tax_space                450
tax_paid                 268
estimated_tax_bill        94
has_luxury_tax_source    776
dtype: int64


In [11]:
from pathlib import Path

# Show the directory Python is currently executing from.
print("Current working directory:", Path.cwd())

# Check whether the expected CSV exists relative to that directory.
print("CSV exists:", Path("data/canonical/team_seasons.csv").exists())

Current working directory: /Users/jonathantran/Desktop/nba_economics/notebooks
CSV exists: False


In [13]:
from pathlib import Path
import pandas as pd

# Find canonical CSVs related to salaries.
canonical_dir = Path("../data/canonical")

salary_files = [
    p.name for p in canonical_dir.glob("*.csv")
    if "salar" in p.name.lower()
]

print("Salary-related canonical files:", salary_files)

Salary-related canonical files: ['minimum_salary_rules.csv', 'player_salaries.csv', 'rookie_salary_scale.csv', 'player_salaries_unresolved.csv']


In [14]:
import pandas as pd

# Load the two C07 outputs.
c07 = pd.read_csv("../data/canonical/player_salaries.csv")
unresolved = pd.read_csv("../data/canonical/player_salaries_unresolved.csv")

# Verify record counts against our previous validation.
print("C07 salary records:", len(c07))
print("Unresolved salary records:", len(unresolved))

# Inspect available fields before deciding whether additional checks are needed.
print("\nC07 columns:")
print(c07.columns.tolist())

print("\nUnresolved columns:")
print(unresolved.columns.tolist())

# Identify missing values in the canonical salary table.
print("\nC07 missing values:")
print(c07.isna().sum().loc[lambda x: x > 0])

C07 salary records: 11826
Unresolved salary records: 380

C07 columns:
['player_id', 'franchise_id', 'season', 'salary', 'source_player_name', 'source_team_name']

Unresolved columns:
['player', 'team', 'season', 'salary', 'player_id', 'franchise_id']

C07 missing values:
Series([], dtype: int64)


In [15]:
from pathlib import Path
import pandas as pd

# List available canonical files so we can identify C08's exact filename.
canonical_dir = Path("../data/canonical")

for file in sorted(canonical_dir.glob("*.csv")):
    if "award" in file.name.lower():
        print(file.name)

player_awards.csv


In [16]:
import pandas as pd

# Load the canonical awards table.
c08 = pd.read_csv("../data/canonical/player_awards.csv")

# Confirm its size and inspect the schema.
print("C08 rows:", len(c08))
print("\nColumns:")
print(c08.columns.tolist())

# Check for missing values in any field.
print("\nMissing values:")
print(c08.isna().sum().loc[lambda x: x > 0])

# Check whether any entire records are duplicated.
print("\nExact duplicate rows:", c08.duplicated().sum())

# Identify the historical season coverage.
if "season" in c08.columns:
    print("\nSeason range:", c08["season"].min(), "to", c08["season"].max())
    print("Distinct seasons:", c08["season"].nunique())

C08 rows: 2570

Columns:
['season', 'source_player_name', 'player_position', 'player_id', 'award_type', 'source_table', 'team_selection', 'age', 'team', 'points_won', 'points_max', 'vote_share', 'position', 'year', 'voting_rank', 'first_place_votes', 'award_winner', 'first_team_votes', 'second_team_votes', 'all_star_appearances', 'votes', 'selection_type', 'player_status', 'conference']

Missing values:
player_position         2180
team_selection          1657
age                     1071
team                     390
points_won              1071
points_max              1071
vote_share              1071
position                1626
year                     390
voting_rank             1594
first_place_votes       1594
award_winner            1594
first_team_votes        2310
second_team_votes       2310
all_star_appearances    1889
votes                   2386
selection_type          1889
player_status           2476
conference              1889
dtype: int64

Exact duplicate rows: 0

Sea

In [17]:
# Check that essential identification fields are populated.
required = ["season", "player_id", "award_type", "source_table"]

print("Missing required identifiers:")
print(c08[required].isna().sum())

# Check award-type distribution to understand the table's structure.
print("\nAward types:")
print(c08["award_type"].value_counts())

# Check duplicates within the source award tables.
# This is a diagnostic candidate key, not necessarily the final grain.
candidate_key = ["season", "player_id", "award_type", "source_table"]

print("\nDuplicate candidate keys:")
print(c08.duplicated(subset=candidate_key).sum())

Missing required identifiers:
season          0
player_id       0
award_type      0
source_table    0
dtype: int64

Award types:
award_type
all_star         681
dpoy             426
all_nba          390
mvp              352
all_defensive    263
all_rookie       260
roy              198
Name: count, dtype: int64

Duplicate candidate keys:
0


In [18]:
import pandas as pd

# Load both C09 reference tables.
rookie = pd.read_csv("../data/canonical/rookie_salary_scale.csv")
minimum = pd.read_csv("../data/canonical/minimum_salary_rules.csv")

for name, df in [
    ("Rookie salary scale", rookie),
    ("Minimum salary rules", minimum)
]:
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum().loc[lambda x: x > 0])


Rookie salary scale
Rows: 775
Columns: ['draft_pick', 'year_1_salary', 'year_2_salary', 'year_3_option_salary', 'year_4_option_increase_pct', 'qualifying_offer_increase_pct', 'season', 'year']
Exact duplicate rows: 0
Missing values:
Series([], dtype: int64)

Minimum salary rules
Rows: 286
Columns: ['season', 'year', 'cba_id', 'years_of_service', 'minimum_salary']
Exact duplicate rows: 0
Missing values:
Series([], dtype: int64)


In [19]:
import pandas as pd

# Load the canonical season-level table.
c01 = pd.read_csv("../data/canonical/seasons.csv")

# Identify threshold-related columns in C01.
threshold_cols = [
    col for col in c01.columns
    if "tax" in col.lower() or "threshold" in col.lower()
]

print("C01 rows:", len(c01))
print("\nTax/threshold columns:")
print(threshold_cols)

# Show the available season-level threshold data.
print("\nC01 threshold preview:")
print(c01[["season"] + threshold_cols].head(10).to_string(index=False))

C01 rows: 26

Tax/threshold columns:
['Luxury Tax', 'Non-Taxpayer MLE', 'Taxpayer MLE']

C01 threshold preview:
   season  Luxury Tax  Non-Taxpayer MLE  Taxpayer MLE
2000-2001         NaN           2250000           NaN
2001-2002         NaN           4538000           NaN
2002-2003  52880000.0           4546000           NaN
2003-2004  54556722.0           4917000           NaN
2004-2005         NaN           4903000           NaN
2005-2006  61700000.0           5000000           NaN
2006-2007  65420000.0           5215000           NaN
2007-2008  67865000.0           5356000           NaN
2008-2009  71150000.0           5585000           NaN
2009-2010  69920000.0           5854000           NaN


In [20]:
from pathlib import Path
import pandas as pd

canonical_dir = Path("../data/canonical")

# List each CSV and its columns without printing the entire dataset.
for path in sorted(canonical_dir.glob("*.csv")):
    df = pd.read_csv(path, nrows=0)

    print(f"\n{path.name}")
    print(", ".join(df.columns))


franchise_season_aliases.csv
franchise_id, season, team_abbreviation, full_team_name, short_team_name

franchises.csv
franchise_id, current_team_name, current_abbreviation

leadership_tenures.csv
franchise_id, leader_name, role, start_date, end_date, start_season, end_season, source_table

minimum_salary_rules.csv
season, year, cba_id, years_of_service, minimum_salary

player_awards.csv
season, source_player_name, player_position, player_id, award_type, source_table, team_selection, age, team, points_won, points_max, vote_share, position, year, voting_rank, first_place_votes, award_winner, first_team_votes, second_team_votes, all_star_appearances, votes, selection_type, player_status, conference

player_salaries.csv
player_id, franchise_id, season, salary, source_player_name, source_team_name

player_salaries_unresolved.csv
player, team, season, salary, player_id, franchise_id

player_team_seasons.csv
player_id, franchise_id, season, age, height_inches, weight_lbs, games_played, point

In [21]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Load canonical tables
# --------------------------------------------------

base = Path("../data/canonical")

files = {
    "C01": "seasons.csv",
    "C02": "franchises.csv",
    "C03": "players.csv",
    "C04": "team_seasons.csv",
    "C05": "player_team_seasons.csv",
    "C06": "leadership_tenures.csv",
    "C07": "player_salaries.csv",
    "C08": "player_awards.csv",
    "C09a": "rookie_salary_scale.csv",
    "C09b": "minimum_salary_rules.csv",
    "Aliases": "franchise_season_aliases.csv",
    "Unresolved": "player_salaries_unresolved.csv",
}

# Dictionary comprehension loads each CSV into a DataFrame.
tables = {
    name: pd.read_csv(base / filename)
    for name, filename in files.items()
}

# --------------------------------------------------
# 2. Expected row counts
# --------------------------------------------------

expected = {
    "C01": 26,
    "C02": 30,
    "C03": 2584,
    "C04": 776,
    "C05": 12810,
    "C06": 251,
    "C07": 11826,
    "C08": 2570,
    "C09a": 775,
    "C09b": 286,
    "Aliases": 776,
    "Unresolved": 380,
}

print("=== ROW COUNT AUDIT ===")

for name, expected_rows in expected.items():
    actual = len(tables[name])
    status = "PASS" if actual == expected_rows else "REVIEW"
    print(f"{name:12} {actual:6} / {expected_rows:6}  {status}")


# --------------------------------------------------
# 3. Primary-key uniqueness
# --------------------------------------------------

print("\n=== KEY UNIQUENESS AUDIT ===")

# Only include keys whose grain is established.
keys = {
    "C01": ["season"],
    "C02": ["franchise_id"],
    "C03": ["player_id"],
    "C04": ["franchise_id", "season"],
    "C05": ["player_id", "franchise_id", "season"],
    "C07": ["player_id", "franchise_id", "season"],
    "C08": ["season", "player_id", "award_type", "source_table"],
    "C09a": ["season", "draft_pick"],
    "C09b": ["season", "years_of_service"],
    "Aliases": ["franchise_id", "season"],
}

for name, columns in keys.items():
    df = tables[name]

    duplicates = df.duplicated(subset=columns).sum()
    missing_keys = df[columns].isna().any(axis=1).sum()

    status = (
        "PASS"
        if duplicates == 0 and missing_keys == 0
        else "REVIEW"
    )

    print(
        f"{name:12} "
        f"duplicates={duplicates:5} "
        f"missing_keys={missing_keys:5} "
        f"{status}"
    )


# --------------------------------------------------
# 4. Foreign-key integrity
# --------------------------------------------------

print("\n=== FOREIGN-KEY AUDIT ===")

def check_fk(child_name, child_column, parent_name, parent_column):
    """
    Count non-null child identifiers that do not exist
    in the corresponding parent table.
    """

    child = tables[child_name][child_column].dropna()
    parent = set(tables[parent_name][parent_column].dropna())

    unmatched = (~child.isin(parent)).sum()

    status = "PASS" if unmatched == 0 else "REVIEW"

    print(
        f"{child_name}.{child_column} -> "
        f"{parent_name}.{parent_column}: "
        f"{unmatched} unmatched [{status}]"
    )


# Franchise references.
for name in ["C04", "C05", "C06", "C07", "Aliases"]:
    check_fk(name, "franchise_id", "C02", "franchise_id")

# Player references.
for name in ["C05", "C07", "C08"]:
    check_fk(name, "player_id", "C03", "player_id")

# Season references.
for name in [
    "C04", "C05", "C07", "C08",
    "C09a", "C09b", "Aliases"
]:
    check_fk(name, "season", "C01", "season")


# --------------------------------------------------
# 5. Composite franchise-season relationships
# --------------------------------------------------

print("\n=== FRANCHISE-SEASON RELATIONSHIPS ===")

# Build sets of (franchise_id, season) tuples.
# Tuples let us compare two-column keys together.
team_season_keys = set(
    map(tuple, tables["C04"][["franchise_id", "season"]].to_numpy())
)

for name in ["C05", "C07", "Aliases"]:

    child_keys = set(
        map(tuple, tables[name][["franchise_id", "season"]].to_numpy())
    )

    unmatched = len(child_keys - team_season_keys)

    status = "PASS" if unmatched == 0 else "REVIEW"

    print(
        f"{name} -> C04: "
        f"{unmatched} unmatched franchise-season keys [{status}]"
    )


# --------------------------------------------------
# 6. Salary coverage against roster observations
# --------------------------------------------------

print("\n=== C07 SALARY COVERAGE ===")

roster = tables["C05"]
salaries = tables["C07"]

# Compare complete player/franchise/season combinations.
roster_keys = set(
    map(
        tuple,
        roster[["player_id", "franchise_id", "season"]].to_numpy()
    )
)

salary_keys = set(
    map(
        tuple,
        salaries[["player_id", "franchise_id", "season"]].to_numpy()
    )
)

matched = len(roster_keys & salary_keys)
unmatched = len(roster_keys - salary_keys)
coverage = 100 * matched / len(roster_keys)

print("Roster observations:", len(roster_keys))
print("Matched salaries:", matched)
print("Unmatched roster observations:", unmatched)
print(f"Coverage: {coverage:.2f}%")

print("\n=== AUDIT COMPLETE ===")

=== ROW COUNT AUDIT ===
C01              26 /     26  PASS
C02              30 /     30  PASS
C03            2584 /   2584  PASS
C04             776 /    776  PASS
C05           12810 /  12810  PASS
C06             251 /    251  PASS
C07           11826 /  11826  PASS
C08            2570 /   2570  PASS
C09a            775 /    775  PASS
C09b            286 /    286  PASS
Aliases         776 /    776  PASS
Unresolved      380 /    380  PASS

=== KEY UNIQUENESS AUDIT ===
C01          duplicates=    0 missing_keys=    0 PASS
C02          duplicates=    0 missing_keys=    0 PASS
C03          duplicates=    0 missing_keys=    0 PASS
C04          duplicates=    0 missing_keys=    0 PASS
C05          duplicates=    0 missing_keys=    0 PASS
C07          duplicates=    0 missing_keys=    0 PASS
C08          duplicates=    0 missing_keys=    0 PASS
C09a         duplicates=    0 missing_keys=    0 PASS
C09b         duplicates=    0 missing_keys=    0 PASS
Aliases      duplicates=    0 missing_ke